# FeederFlow Y-bus and Power-Flow Validation Walkthrough

This notebook mirrors the current correctness tests and prints key diagnostics:
1. **Y-bus correctness** (`Y`, `Y_NS`, `Y_SS`, `YL`) against MATLAB fixtures.
2. **Power-flow correctness** against MATLAB/OpenDSS with current solver-available keys and tolerances.
3. **Regulator secondary voltage reconstruction** (IEEE123).

Important outputs include matrix dimensions, `nnz`, key-overlap ratios, max absolute voltage deltas, convergence history, and regulator-node diffs.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

using Test
using LinearAlgebra
using SparseArrays
using FeederFlow

include(joinpath(@__DIR__, "..", "test", "test_support.jl"))

println("Active project: ", Base.active_project())
println("Julia version: ", VERSION)

  Activating project at `c:\Users\hoang\OneDrive - Massachusetts Institute of Technology\1. MIT\2. Projects\4. Dist OPF\multiphase_modelling\FeederFlow.jl`


Active project: c:\Users\hoang\OneDrive - Massachusetts Institute of Technology\1. MIT\2. Projects\4. Dist OPF\multiphase_modelling\FeederFlow.jl\Project.toml
Julia version: 1.12.5


In [2]:
function assert_sparse_matches_fixture_verbose(actual, actual_rows, actual_cols, fixture_obj, expected_rows, expected_cols; label, rtol = 1e-8, atol = 1e-10)
    expected = sparse_from_fixture(fixture_obj)
    reordered = reordered_actual_rect(actual, actual_rows, actual_cols, expected_rows, expected_cols)
    diff = matrix_max_abs_diff(reordered, expected)
    println("[$label] size=", size(reordered), " nnz(actual)=", nnz(reordered), " nnz(expected)=", nnz(expected), " max_abs_diff=", diff)
    @test size(reordered) == size(expected)
    @test isapprox(reordered, expected; rtol = rtol, atol = atol)
end

function assert_voltage_map_matches_verbose(actual, expected; label, atol = 1e-8, rtol = 1e-8, min_actual_key_coverage = 1.0)
    shared_keys = intersect(keys(actual), keys(expected))
    @test !isempty(shared_keys)
    coverage = isempty(actual) ? 1.0 : length(shared_keys) / length(keys(actual))
    println("[$label] actual_keys=", length(keys(actual)), " expected_keys=", length(keys(expected)), " shared=", length(shared_keys), " coverage=", round(coverage, digits=4))
    if !isempty(actual)
        @test coverage >= min_actual_key_coverage
    end
    maxdiff = max_voltage_diff(actual, expected)
    println("[$label] max_abs_diff=", maxdiff, " (atol=", atol, ")")
    if !(maxdiff <= atol || isapprox(maxdiff, 0.0; atol = atol, rtol = rtol))
        @info("$label mismatch", max_abs_diff = maxdiff, report = mismatch_report(actual, expected))
    end
    @test maxdiff <= atol || isapprox(maxdiff, 0.0; atol = atol, rtol = rtol)
end

function summarize_history(history, label; tol = 1e-5)
    println("[$label] iterations=", length(history), " final_residual=", isempty(history) ? missing : history[end])
    @test !isempty(history)
    @test history[end] <= tol
    @test all(diff(history) .<= 0.0)
end

summarize_history (generic function with 1 method)

## 1) Y-bus correctness walkthrough (mirrors `test_ybus_correctness.jl`)

Checks performed per feeder:
- Network/slack key-set parity with fixture ordering sets.
- Matrix dimensions and Y symmetry.
- Capacitor presence/absence expectation.
- Sparse matrix value checks for `Y`, `Y_NS`, `Y_SS`, `YL`.

In [3]:
@testset "Y-bus correctness walkthrough - IEEE37" begin
    fixture = load_fixture("ieee37_matlab_reference.json")
    network = parse_file(IEEE37_DSS)
    ybus = build_y(network)
    noload = compute_no_load(ybus)
    loads = build_load_model(network, ybus, noload)

    actual_network_order = busphase_key.(ybus.network_order)
    actual_slack_order = busphase_key.(ybus.slack_order)
    expected_network_order = json_string_vector(fixture["network_order"])
    expected_slack_order = json_string_vector(fixture["slack_order"])

    println("[IEEE37] network nodes=", length(actual_network_order), " slack nodes=", length(actual_slack_order), " capacitors=", length(network.capacitors))
    @test Set(actual_network_order) == Set(expected_network_order)
    @test Set(actual_slack_order) == Set(expected_slack_order)
    @test size(ybus.Y, 1) == length(ybus.network_order)
    @test size(ybus.Y, 2) == length(ybus.network_order)
    @test size(ybus.Y_NS, 2) == 3
    @test size(ybus.Y_SS) == (3, 3)
    @test maximum(abs.(Matrix(ybus.Y - transpose(ybus.Y)))) <= 1e-8
    @test isempty(network.capacitors)

    assert_sparse_matches_fixture_verbose(ybus.Y, actual_network_order, actual_network_order, fixture["y"], expected_network_order, expected_network_order; label = "IEEE37 Y")
    assert_sparse_matches_fixture_verbose(ybus.Y_NS, actual_network_order, actual_slack_order, fixture["y_ns"], expected_network_order, expected_slack_order; label = "IEEE37 Y_NS")
    assert_sparse_matches_fixture_verbose(ybus.Y_SS, actual_slack_order, actual_slack_order, fixture["y_ss"], expected_slack_order, expected_slack_order; label = "IEEE37 Y_SS")
    assert_sparse_matches_fixture_verbose(loads.YL, actual_network_order, actual_network_order, fixture["yl"], expected_network_order, expected_network_order; label = "IEEE37 YL")
end

@testset "Y-bus correctness walkthrough - IEEE123" begin
    fixture = load_fixture("ieee123_matlab_reference.json")
    network = parse_file(IEEE123_DSS)
    ybus = build_y(network)
    noload = compute_no_load(ybus)
    loads = build_load_model(network, ybus, noload)

    actual_network_order = busphase_key.(ybus.network_order)
    actual_slack_order = busphase_key.(ybus.slack_order)
    expected_network_order = json_string_vector(fixture["network_order"])
    expected_slack_order = json_string_vector(fixture["slack_order"])

    println("[IEEE123] network nodes=", length(actual_network_order), " slack nodes=", length(actual_slack_order), " capacitors=", length(network.capacitors))
    @test Set(actual_network_order) == Set(expected_network_order)
    @test Set(actual_slack_order) == Set(expected_slack_order)
    @test size(ybus.Y, 1) == length(ybus.network_order)
    @test size(ybus.Y, 2) == length(ybus.network_order)
    @test size(ybus.Y_NS, 2) == 3
    @test size(ybus.Y_SS) == (3, 3)
    @test maximum(abs.(Matrix(ybus.Y - transpose(ybus.Y)))) <= 1e-8
    @test !isempty(network.capacitors)

    assert_sparse_matches_fixture_verbose(ybus.Y, actual_network_order, actual_network_order, fixture["y"], expected_network_order, expected_network_order; label = "IEEE123 Y", atol = 1e-8)
    assert_sparse_matches_fixture_verbose(ybus.Y_NS, actual_network_order, actual_slack_order, fixture["y_ns"], expected_network_order, expected_slack_order; label = "IEEE123 Y_NS", atol = 1e-8)
    assert_sparse_matches_fixture_verbose(ybus.Y_SS, actual_slack_order, actual_slack_order, fixture["y_ss"], expected_slack_order, expected_slack_order; label = "IEEE123 Y_SS", atol = 1e-8)
    assert_sparse_matches_fixture_verbose(loads.YL, actual_network_order, actual_network_order, fixture["yl"], expected_network_order, expected_network_order; label = "IEEE123 YL")
end

┌ Warning: Approximating OpenDSS Load Model=4 as PQ for benchmark compatibility
└ @ FeederFlow C:\Users\hoang\OneDrive - Massachusetts Institute of Technology\1. MIT\2. Projects\4. Dist OPF\multiphase_modelling\FeederFlow.jl\src\loads.jl:51


[IEEE37] network nodes=111 slack nodes=3 capacitors=0
[IEEE37 Y] size=(111, 111) nnz(actual)=981 nnz(expected)=981 max_abs_diff=6.355287432313019e-14
[IEEE37 Y_NS] size=(111, 3) nnz(actual)=9 nnz(expected)=9 max_abs_diff=0.0
[IEEE37 Y_SS] size=(3, 3) nnz(actual)=9 nnz(expected)=9 max_abs_diff=0.0
[IEEE37 YL] size=(111, 111) nnz(actual)=27 nnz(expected)=27 max_abs_diff=0.0
Test Summary:                          | Pass  Total   Time
Y-bus correctness walkthrough - IEEE37 |   16     16  12.2s
[IEEE123] network nodes=262 slack nodes=3 capacitors=4
[IEEE123 Y] size=(262, 262) nnz(actual)=1992 nnz(expected)=1992 max_abs_diff=2.4361419939111784e-5
[IEEE123 Y_NS] size=(262, 3) nnz(actual)=9 nnz(expected)=9 max_abs_diff=7.809377196826238e-10
[IEEE123 Y_SS] size=(3, 3) nnz(actual)=9 nnz(expected)=9 max_abs_diff=7.392486076486866e-12
[IEEE123 YL] size=(262, 262) nnz(actual)=25 nnz(expected)=25 max_abs_diff=0.0
Test Summary:                           | Pass  Total  Time
Y-bus correctness walkthrou

Test.DefaultTestSet("Y-bus correctness walkthrough - IEEE123", Any[], 16, false, false, true, 1.775253698663e9, 1.775253698744e9, false, "In[3]", Random.Xoshiro(0x8e542c7397f5da2c, 0xb0ace3194b7872ad, 0x0a95bff15d61aefb, 0x25f1e5bce6a87d02, 0xdee4dda09be30828))

## 2) Power-flow correctness walkthrough (mirrors current `test_power_flow_correctness.jl`)

Checks performed per feeder:
- No-load and final voltage agreement against MATLAB fixture with current tolerances.
- Final voltage agreement against OpenDSS export with current tolerances.
- Presence of critical regulator secondary keys.
- Convergence status and residual monotonicity.

In [4]:
@testset "Power-flow correctness walkthrough - IEEE37" begin
    fixture = load_fixture("ieee37_matlab_reference.json")
    bundle = solve_case(IEEE37_DSS; max_iter = 10, tol = 1e-5)

    actual_noload = actual_noload_map(bundle.ybus, bundle.noload)
    expected_noload = fixture_voltage_map(fixture["noload_phase_voltages"])
    assert_voltage_map_matches_verbose(actual_noload, expected_noload; label = "IEEE37 no-load", atol = 5e-3)

    actual_final = actual_voltage_map(bundle.result.phase_voltages)
    expected_final = fixture_voltage_map(fixture["final_phase_voltages"])
    assert_voltage_map_matches_verbose(actual_final, expected_final; label = "IEEE37 MATLAB final", atol = 1e-2)

    dss_final = parse_opendss_voltages(IEEE37_DSS_VOLTAGES)
    assert_voltage_map_matches_verbose(actual_final, dss_final; label = "IEEE37 OpenDSS final", atol = 2e-2)

    for key in ("799r.1", "799r.2", "799r.3")
        @test haskey(actual_final, key)
    end
    @test bundle.result.converged == Bool(fixture["converged"])
    summarize_history(bundle.result.history, "IEEE37 history")
end

@testset "Power-flow correctness walkthrough - IEEE123" begin
    fixture = load_fixture("ieee123_matlab_reference.json")
    bundle = solve_case(IEEE123_DSS; max_iter = 10, tol = 1e-5)

    actual_noload = actual_noload_map(bundle.ybus, bundle.noload)
    expected_noload = fixture_voltage_map(fixture["noload_phase_voltages"])
    assert_voltage_map_matches_verbose(actual_noload, expected_noload; label = "IEEE123 no-load", atol = 2e-4)

    actual_final = actual_voltage_map(bundle.result.phase_voltages)
    expected_final = fixture_voltage_map(fixture["final_phase_voltages"])
    assert_voltage_map_matches_verbose(actual_final, expected_final; label = "IEEE123 MATLAB final", atol = 2e-4)

    dss_final = parse_opendss_voltages(IEEE123_DSS_VOLTAGES)
    assert_voltage_map_matches_verbose(actual_final, dss_final; label = "IEEE123 OpenDSS final", atol = 1e-2)

    for key in ("150r.1", "150r.2", "150r.3", "9r.1", "25r.1", "25r.3", "160r.1", "160r.2", "160r.3")
        @test haskey(actual_final, key)
    end
    @test bundle.result.converged == Bool(fixture["converged"])
    summarize_history(bundle.result.history, "IEEE123 history")
end

┌ Warning: Approximating OpenDSS Load Model=4 as PQ for benchmark compatibility
└ @ FeederFlow C:\Users\hoang\OneDrive - Massachusetts Institute of Technology\1. MIT\2. Projects\4. Dist OPF\multiphase_modelling\FeederFlow.jl\src\loads.jl:51


[IEEE37 no-load] actual_keys=114 expected_keys=114 shared=114 coverage=1.0
[IEEE37 no-load] max_abs_diff=0.0030204132209720354 (atol=0.005)
[IEEE37 MATLAB final] actual_keys=117 expected_keys=117 shared=117 coverage=1.0
[IEEE37 MATLAB final] max_abs_diff=0.007078284624269758 (atol=0.01)
[IEEE37 OpenDSS final] actual_keys=117 expected_keys=117 shared=117 coverage=1.0
[IEEE37 OpenDSS final] max_abs_diff=0.018108819828783013 (atol=0.02)
[IEEE37 history] iterations=6 final_residual=6.533168199863851e-6
Test Summary:                               | Pass  Total  Time
Power-flow correctness walkthrough - IEEE37 |   16     16  1.7s
[IEEE123 no-load] actual_keys=265 expected_keys=265 shared=265 coverage=1.0
[IEEE123 no-load] max_abs_diff=0.00010301079951427833 (atol=0.0002)
[IEEE123 MATLAB final] actual_keys=274 expected_keys=274 shared=274 coverage=1.0
[IEEE123 MATLAB final] max_abs_diff=9.309604282757634e-5 (atol=0.0002)
[IEEE123 OpenDSS final] actual_keys=274 expected_keys=278 shared=274 cov

Test.DefaultTestSet("Power-flow correctness walkthrough - IEEE123", Any[], 22, false, false, true, 1.775253700611e9, 1.775253700625e9, false, "In[4]", Random.Xoshiro(0x8e542c7397f5da2c, 0xb0ace3194b7872ad, 0x0a95bff15d61aefb, 0x25f1e5bce6a87d02, 0xdee4dda09be30828))

## 3) Regulator secondary voltage walkthrough (mirrors `test_regulator_secondary.jl`)

This section reconstructs IEEE123 regulator-secondary voltages from network data and compares to MATLAB fixture on `*.r.*` keys.

In [5]:
@testset "Regulator secondary walkthrough - IEEE123" begin
    fixture = load_fixture("ieee123_matlab_reference.json")
    bundle = solve_case(IEEE123_DSS; max_iter = 10, tol = 1e-5)
    ybus = bundle.ybus
    result = bundle.result
    network = bundle.network
    base = network.base

    regulator_config = Dict(
        "150" => ("150r", "149", Dict(1 => 7, 2 => 7, 3 => 7)),
        "9"   => ("9r",   "14",  Dict(1 => -1)),
        "25"  => ("25r",  "26",  Dict(1 => 0, 3 => -1)),
        "160" => ("160r", "67",  Dict(1 => 8, 2 => 1, 3 => 5)),
    )

    ztreg_val(primary_bus) = primary_bus == "150" ? complex(0.0, (base.Sbase / 5_000_000) * 3 * 0.00001) : complex(0.0, (base.Sbase / 2_000_000) * 0.0001)
    get_v(bus, phase) = begin
        bp = BusPhase(bus, phase)
        idx = get(ybus.all_index, bp, 0)
        idx > 0 ? result.voltages[idx] : 0.0 + 0im
    end

    sec_voltages = Dict{String,ComplexF64}()
    for (primary, (sec_bus, downstream, taps)) in regulator_config
        zt = ztreg_val(primary)
        phases = sort(collect(keys(taps)))

        Av = zeros(ComplexF64, 3, 3)
        Ai = zeros(ComplexF64, 3, 3)
        Zrg = zeros(ComplexF64, 3, 3)
        iAv = zeros(ComplexF64, 3, 3)
        for ph in phases
            ar = 1.0 - 0.00625 * taps[ph]
            Av[ph, ph] = ar
            iAv[ph, ph] = 1.0 / ar
            Ai[ph, ph] = 1.0 / ar
            Zrg[ph, ph] = zt / ar
        end

        relevant_line = nothing
        for line in network.lines
            buses_in_line = Set([line.from.bus, line.to.bus])
            if sec_bus in buses_in_line && downstream in buses_in_line
                relevant_line = line
                break
            end
        end
        @test relevant_line !== nothing
        line = relevant_line

        rmat = line.rmatrix
        xmat = line.xmatrix
        cmat = line.cmatrix
        len = line.length
        line_phases = line.phases

        n = size(rmat, 1)
        Z_ohm = zeros(ComplexF64, n, n)
        for i in 1:n, j in 1:n
            Z_ohm[i, j] = rmat[i, j] + im * xmat[i, j]
        end
        Z_ohm *= len
        Z_pu = Z_ohm / base.Zbase
        Y_series = inv(Z_pu)

        omega = 2.0 * pi * 60.0
        Y_sh_mat = zeros(ComplexF64, n, n)
        for i in 1:n, j in 1:n
            Y_sh_mat[i, j] = im * omega * cmat[i, j] * len * 1e-9
        end
        Y_sh_pu = Y_sh_mat / base.Ybase

        nn = zeros(ComplexF64, 3, 3)
        nm = zeros(ComplexF64, 3, 3)
        for (i_local, i_global) in enumerate(line_phases), (j_local, j_global) in enumerate(line_phases)
            nn[i_global, j_global] += Y_series[i_local, j_local] + 0.5 * Y_sh_pu[i_local, j_local]
            nm[i_global, j_global] += Y_series[i_local, j_local]
        end

        F = I + nn * iAv * Zrg
        Yn = Ai * inv(F) * nn * iAv
        Ym = Ai * inv(F) * nm

        V_primary = zeros(ComplexF64, 3)
        V_downstream = zeros(ComplexF64, 3)
        for ph in 1:3
            V_primary[ph] = get_v(primary, ph)
            V_downstream[ph] = get_v(downstream, ph)
        end

        V_sec = iAv * (V_primary - Zrg * Ai * (Yn * V_primary - Ym * V_downstream))
        for ph in phases
            sec_voltages["$sec_bus.$ph"] = V_sec[ph]
        end
    end

    expected = fixture_voltage_map(fixture["final_phase_voltages"])
    expected_reg = Dict(k => v for (k, v) in expected if occursin("r.", k))
    @test Set(keys(sec_voltages)) == Set(keys(expected_reg))

    diffs = Float64[]
    for (key, v_expected) in expected_reg
        v_actual = sec_voltages[key]
        diff = abs(v_actual - v_expected)
        push!(diffs, diff)
        @test diff <= 1e-4
    end
    println("[Regulator secondary] keys=", length(diffs), " max_abs_diff=", maximum(diffs), " mean_abs_diff=", sum(diffs)/length(diffs))
end

[Regulator secondary] keys=9 max_abs_diff=8.578425671924304e-5 mean_abs_diff=3.851664163500135e-5
Test Summary:                             | Pass  Total  Time
Regulator secondary walkthrough - IEEE123 |   14     14  0.5s


Test.DefaultTestSet("Regulator secondary walkthrough - IEEE123", Any[], 14, false, false, true, 1.775253700753e9, 1.775253701228e9, false, "In[5]", Random.Xoshiro(0x8e542c7397f5da2c, 0xb0ace3194b7872ad, 0x0a95bff15d61aefb, 0x25f1e5bce6a87d02, 0xdee4dda09be30828))

## 4) End-to-end summary

If this final cell passes, the notebook is aligned with current tests and validations.

In [6]:
@test true
println("Notebook validation definitions loaded. Run all cells top-to-bottom to execute the walkthrough.")

Notebook validation definitions loaded. Run all cells top-to-bottom to execute the walkthrough.
